# MLP y Regresión logística

En este jupyternotebook se va a entrenar por un lado una regresión logística y un MLP para comparar sus soluciones a la hora de diferenciar la señal del fondo en dos casos, uno con las variables solo PRI y otro con las variables PRI y DER. 

In [ ]:
import numpy as np
import pandas as pd
# ----------------------------------
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
# -----------------------------------
import tensorflow as tf
import keras
from keras import layers
from sklearn.model_selection import train_test_split
# -----------------------------------
import matplotlib.pyplot as plt

## Carga de datos

Para desarrollar tanto la regresión logística como el MLP se va a hacer uso de los tres csv . El de entrenamiento para entrenar, el de Ajuste para ajustar los hiperparámetros y el de Test para analizar el resultado final de cada modelo entrenado.

In [3]:
Entrenamiento = pd.read_csv("Entrenamiento.csv")
Ajuste = pd.read_csv("Ajuste.csv")
Test = pd.read_csv("Test.csv")

Estos csv ya se han sometido a una normalización y limpieza de datos (mirar el EDA), por lo que solo es necesario ajustar los parametros para poder manipularlos en los diferentes modelos.

Los valores -999 no son valores que se tengan que eliminar como ya se ha observado en el SOM, por tanto se van a convertir en valores nulos para que no condicionen los resultados.

In [4]:
Entrenamiento = Entrenamiento.replace(np.nan, 0)
Ajuste = Ajuste.replace(np.nan, 0)
Test = Test.replace(np.nan, 0)

Se va a intercambiar los valores de señal y fondo por 0 y 1 para que la matriz Y sea numérica

In [ ]:
Entrenamiento['Label'] = Entrenamiento["Label"].replace({"s":1, "b":0}).astype(int)
Ajuste["Label"] = Ajuste["Label"].replace({"s":1, "b":0}).astype(int)
Test["Label"] = Test["Label"].replace({"s":1, "b":0}).astype(int)

# Regresión logística

## Preparación de datos

Como se ha descrito anteriormente, al regresión logística se va a calcular respecto de los dos subgrupos PRI y PRI + DER junto con las columna del label que dice si es fondo o señal y los pesos para el AMS.

In [6]:
Entrenamiento_DER = Entrenamiento.drop(columns=["EventId", "Weight", "KaggleSet", "Label", "KaggleWeight"])
Entrenamiento_PRI = Entrenamiento_DER[["PRI_tau_pt", "PRI_tau_eta", "PRI_tau_phi","PRI_lep_pt","PRI_lep_eta","PRI_lep_phi","PRI_met","PRI_met_phi","PRI_met_sumet","PRI_jet_num","PRI_jet_leading_pt","PRI_jet_leading_eta","PRI_jet_leading_phi","PRI_jet_subleading_pt", "PRI_jet_subleading_eta","PRI_jet_subleading_phi","PRI_jet_all_pt"]].copy().reset_index(drop=True)
Y_E = Entrenamiento["Label"]
W_E = Entrenamiento["KaggleWeight"]

In [7]:
Ajuste_DER = Ajuste.drop(columns=["EventId", "Weight", "KaggleSet", "Label", "KaggleWeight"])
Ajuste_PRI = Ajuste_DER[["PRI_tau_pt", "PRI_tau_eta", "PRI_tau_phi","PRI_lep_pt","PRI_lep_eta","PRI_lep_phi","PRI_met","PRI_met_phi","PRI_met_sumet","PRI_jet_num","PRI_jet_leading_pt","PRI_jet_leading_eta","PRI_jet_leading_phi","PRI_jet_subleading_pt", "PRI_jet_subleading_eta","PRI_jet_subleading_phi","PRI_jet_all_pt"]].copy().reset_index(drop=True)
Y_A = Ajuste["Label"]
W_A = Ajuste["KaggleWeight"]

In [8]:
Test_DER = Test.drop(columns=["EventId", "Weight", "KaggleSet", "Label", "KaggleWeight"])
Test_PRI = Test_DER[["PRI_tau_pt", "PRI_tau_eta", "PRI_tau_phi","PRI_lep_pt","PRI_lep_eta","PRI_lep_phi","PRI_met","PRI_met_phi","PRI_met_sumet","PRI_jet_num","PRI_jet_leading_pt","PRI_jet_leading_eta","PRI_jet_leading_phi","PRI_jet_subleading_pt", "PRI_jet_subleading_eta","PRI_jet_subleading_phi","PRI_jet_all_pt"]].copy().reset_index(drop=True)
Y_T = Test["Label"]
W_T = Test["KaggleWeight"]

## Entrenamiento de la Regresión logística

La regresión logística es un modelo de aprendizaje supervisado lineal con una sola capa oculta.

In [ ]:
def calcular_ams(y, prob, pesos, umbral):           # el AMS permite ver cómo de acertado fue (cuanto más alto mejor)
    c = 10
    seleccionados = prob >= umbral                  # Genera un vector true false, siendo true si el evento es cogido por señal, es decir pasa el filtro del umbral
    s = pesos[seleccionados & (y == 1)].sum()       # Suma la cantidad de eventos que sean señal tanto en el modelo como de verdad
    b = pesos[seleccionados & (y == 0)].sum()

    AMS = np.sqrt(2*((s+b+c)*np.log(1+(s/(b+c)))-s))
    return AMS

def buscar_mejor_umbral(y, prob, pesos):
    umbrales = np.quantile(prob, np.linspace(0.50, 0.99, 200))
    lista_ams = [calcular_ams(y, prob, pesos, u) for u in umbrales]
    mejor = np.argmax(lista_ams)
    return umbrales[mejor], lista_ams[mejor]

In [11]:
def regresion_logistica(Entrenamiento, Y_E, Ajuste, Y_A, W_A, Test, Y_T, W_T, C=np.inf):
    # 1) Entrenar (pasos 1-3 del pseudocódigo)
    modelo = LogisticRegression(solver="newton-cholesky", C=C, tol=1e-7, max_iter=20)   # Crea el modelo
    modelo.fit(Entrenamiento, Y_E)                  # Entrena hasta que deja de cambiar o max_iter

    print("Iteraciones:", modelo.n_iter_)           # cuántas vueltas dio el bucle
    print(modelo.intercept_, modelo.coef_)          # el vector β final: β₀ y β₁…β_d

    # 2) Elegir el umbral con el mejor AMS en VALIDACIÓN
    p_val = modelo.predict_proba(Ajuste)[:, 1]
    umbral, ams_val = buscar_mejor_umbral(Y_A, p_val, W_A)

    # 3) Aplicar ese umbral al TEST
    p_test = modelo.predict_proba(Test)[:, 1]
    ams_test = calcular_ams(Y_T, p_test, W_T, umbral)
    # auc_test = roc_auc_score(L_T, p_test, sample_weight = W_T)

    print(f"Umbral = {umbral:.3f} | AMS val = {ams_val:.3f} | AMS test = {ams_test:.3f}")
    return modelo, ams_test

In [12]:
modelo_pri, ams_pri = regresion_logistica(Entrenamiento_PRI, Y_E, Ajuste_PRI, Y_A, W_A, Test_PRI, Y_T, W_T)
modelo_der, ams_der = regresion_logistica(Entrenamiento_DER, Y_E, Ajuste_DER, Y_A, W_A, Test_DER, Y_T, W_T)

print(f"AMS PRI / AMS PRI+DER = {ams_pri / ams_der:.1%}")

Iteraciones: [4]
[-0.44785614] [[ 0.57123369 -0.00440606 -0.01019967 -0.21368881 -0.00151402  0.00378634
  -0.35406173  0.01067767  0.01582954 -0.28355078  0.34082926  0.00170435
   0.00075096 -0.23060109  0.00723793 -0.00837861  0.60880778]]
Umbral = 0.327 | AMS val = 1.641 | AMS test = 1.632
Iteraciones: [5]
[-1.45624949] [[ 2.60965978e-01 -4.16818121e-01 -7.48839411e-01  3.87911633e-02
   1.49214794e+00 -1.25215500e+00 -1.08513817e-01  7.38814344e-01
  -9.33243394e-02 -2.15106529e-01  8.65104009e+00  2.37747286e-01
   3.58271897e-01  7.65291570e+00 -2.66074461e-03 -5.36713422e-03
  -5.63358774e+00 -4.35387582e-03  5.74756816e-03 -1.01437206e-01
   3.35643363e-03 -7.08065924e-03 -1.51419194e-01  4.25185254e-01
   2.14100721e-04  6.27506492e-04  2.92990051e-01  8.23981447e-03
  -2.97550693e-03  4.32913421e-01  8.63984481e-01]]
Umbral = 0.435 | AMS val = 1.884 | AMS test = 1.897
AMS PRI / AMS PRI+DER = 86.0%


## Análisis de resultados

Resultados obtenidos:

| Modelo | Iteraciones | Umbral | AMS (Ajuste) | AMS (Test) |
| --- | --- | --- | --- | --- |
| PRI | 4 | 0.327 | 1.641 | 1.632 |
| PRI + DER | 5 | 0.435 | 1.884 | 1.879 |

El umbral óptimo se ha elegido maximizando el AMS en el conjunto de validación (Ajuste) y se ha aplicado sin cambios al conjunto de test. En ambos modelos, el AMS de test es prácticamente igual al de validación (1,632 frente a 1,641, y 1,879 frente a 1,860). Esto indica que el modelo y el umbral generalizan bien a eventos que no se han usado ni para entrenar ni para elegir el umbral, y que no hay sobreajuste.

Añadir las variables DER mejora el AMS de test de 1,632 a 1,879, es decir, un 15 % más. Las variables derivadas, calculadas por los físicos a partir de las primitivas, aportan información que un modelo lineal no puede extraer por sí solo de las variables PRI. Tiene sentido: muchas DER son combinaciones no lineales de las PRI (masas invariantes, cocientes, distancias angulares), y la regresión logística solo puede combinar las variables de forma lineal.

La hipótesis H1 planteaba que el AMS con solo variables PRI sería al menos el 90 % del AMS con todas las variables. Con la regresión logística se obtiene un 86.0 %, así que la hipótesis no se cumple para el modelo lineal.

Este resultado no es definitivo para H1, ya que la regresión logística es solo la línea base. El MLP, al poder aprender relaciones no lineales entre las variables, podría reconstruir parte de la información que aportan las DER a partir de las PRI. Si eso ocurre, el cociente debería acercarse al 90 %.

### Intepretación de los cocientes

Como las variables están normalizadas, el valor absoluto de cada coeficiente indica cuánto influye esa variable en la decisión del modelo, y el signo indica hacia dónde empuja (positivo hacia señal, negativo hacia fondo). Eso no supone que son las variables más determinantes o importantes del conjunto de variables. Además, los ángulos φ (PRI_tau_phi, PRI_lep_phi, PRI_met_phi …) tienen coeficientes casi nulos, entre 0,001 y 0,01. Esto es coherente con la física del problema: el detector es simétrico respecto al eje del haz, así que el ángulo azimutal de una partícula, por sí solo, no distingue la señal del fondo.

### Coeficientes del modelo solo PRI

Intercepto: β₀ = −0,726

| # | Variable | β | Tendencia | 
|---|---|---|---|
| 1 | PRI_tau_pt | 0,571 | hacia señal |
| 2 | PRI_tau_eta | −0,004 | ≈ sin influencia |
| 3 | PRI_tau_phi | −0,010 | ≈ sin influencia |
| 4 | PRI_lep_pt | −0,214 | hacia fondo |
| 5 | PRI_lep_eta | −0,002 | ≈ sin influencia |
| 6 | PRI_lep_phi | 0,004 | ≈ sin influencia |
| 7 | PRI_met | −0,354 | hacia fondo |
| 8 | PRI_met_phi | 0,011 | ≈ sin influencia |
| 9 | PRI_met_sumet | 0,016 | ≈ sin influencia |
| 10 | PRI_jet_num | −0,277 | hacia fondo |
| 11 | PRI_jet_leading_pt | 0,341 | hacia señal |
| 12 | PRI_jet_leading_eta | 0,002 | ≈ sin influencia |
| 13 | PRI_jet_leading_phi | 0,001 | ≈ sin influencia |
| 14 | PRI_jet_subleading_pt | −0,231 | hacia fondo |
| 15 | PRI_jet_subleading_eta | 0,007 | ≈ sin influencia |
| 16 | PRI_jet_subleading_phi | −0,008 | ≈ sin influencia |
| 17 | PRI_jet_all_pt | 0,609 | hacia señal |


### Coeficientes del modelo PRI + DER (con indicadora de DER_mass_MMC)

Intercepto: β₀ = −0,872

| # | Variable | β | Tendencia |
|---|---|---|---|
| 1 | DER_mass_MMC | 0,261 | hacia señal |
| 2 | DER_mass_transverse_met_lep | −0,417 | hacia fondo |
| 3 | DER_mass_vis | −0,749 | hacia fondo |
| 4 | DER_pt_h | 0,039 | ≈ sin influencia |
| 5 | DER_deltaeta_jet_jet | 1,492 | hacia señal |
| 6 | DER_mass_jet_jet | −1,252 | hacia fondo |
| 7 | DER_prodeta_jet_jet | −0,109 | hacia fondo |
| 8 | DER_deltar_tau_lep | 0,739 | hacia señal |
| 9 | DER_pt_tot | −0,093 | hacia fondo |
| 10 | DER_sum_pt | −0,215 | hacia fondo |
| 11 | DER_pt_ratio_lep_tau | 8,651 | hacia señal |
| 12 | DER_met_phi_centrality | 0,238 | hacia señal |
| 13 | DER_lep_eta_centrality | 0,358 | hacia señal |
| 14 | PRI_tau_pt | 7,653 | hacia señal |
| 15 | PRI_tau_eta | −0,003 | ≈ sin influencia |
| 16 | PRI_tau_phi | −0,005 | ≈ sin influencia |
| 17 | PRI_lep_pt | −5,634 | hacia fondo |
| 18 | PRI_lep_eta | −0,004 | ≈ sin influencia |
| 19 | PRI_lep_phi | 0,006 | ≈ sin influencia |
| 20 | PRI_met | −0,101 | hacia fondo |
| 21 | PRI_met_phi | 0,003 | ≈ sin influencia |
| 22 | PRI_met_sumet | −0,007 | ≈ sin influencia |
| 23 | PRI_jet_num | −0,148 | hacia fondo |
| 24 | PRI_jet_leading_pt | 0,425 | hacia señal |
| 25 | PRI_jet_leading_eta | 0,0002 | ≈ sin influencia |
| 26 | PRI_jet_leading_phi | 0,0006 | ≈ sin influencia |
| 27 | PRI_jet_subleading_pt | 0,293 | hacia señal |
| 28 | PRI_jet_subleading_eta | 0,008 | ≈ sin influencia |
| 29 | PRI_jet_subleading_phi | −0,003 | ≈ sin influencia |
| 30 | PRI_jet_all_pt | 0,433 | hacia señal |
| 31 | MMC_no_definida | 0,311 | hacia señal |

Se considera "≈ sin influencia" cuando |β| < 0,05.
El interceptor muestra hacia donde tiende el modelo en el inicio. Que sea negativo significa que tiende hacia el fondo, lo que se corresponde con el modelo ya que tiene más eventos de fondo que de señal.

# MLP

Definición y uso en este proyecto

## Preparación de datos

Se va diferencias dos grupos en entrenamiento, un 80% con el que se hará el entrenamiento y otro 20% para la validación

In [ ]:
Entrenamiento, Validacion = train_test_split(Entrenamiento, test_size=0.2,stratify=Entrenamiento["Label"], random_state=42)

Sin embargo, esta estratificación supone que se tenga que hacer un ajuste en los pesos o kaggleweight de cada uno de los grupos:

In [15]:
def equilibrar(df):
    w = df["KaggleWeight"] / df.groupby("Label")["KaggleWeight"].transform("sum") * (len(df) / 2)
    return w.to_numpy()

Como el MLP se va a hacer respecto de PRI con PRI+DER se vuelven a separar tanto Entrenamiento como validación en otros dos subconjuntos:

In [16]:
# Entrenamiento DER / DER + PRI
Entrenamiento_DER = Entrenamiento.drop(columns=["EventId", "Weight", "KaggleSet", "Label", "KaggleWeight"])
Entrenamiento_PRI = Entrenamiento_DER[["PRI_tau_pt", "PRI_tau_eta", "PRI_tau_phi","PRI_lep_pt","PRI_lep_eta","PRI_lep_phi","PRI_met","PRI_met_phi","PRI_met_sumet","PRI_jet_num","PRI_jet_leading_pt","PRI_jet_leading_eta","PRI_jet_leading_phi","PRI_jet_subleading_pt", "PRI_jet_subleading_eta","PRI_jet_subleading_phi","PRI_jet_all_pt"]].copy().reset_index(drop=True)

# Ya existen Y_E ("label en valores numéricos 0 1")
Y_E = Entrenamiento["Label"]
W_E = equilibrar(Entrenamiento)

In [17]:
Validacion_DER = Validacion.drop(columns=["EventId", "Weight", "KaggleSet", "Label", "KaggleWeight"])
Validacion_PRI = Validacion_DER[["PRI_tau_pt", "PRI_tau_eta", "PRI_tau_phi","PRI_lep_pt","PRI_lep_eta","PRI_lep_phi","PRI_met","PRI_met_phi","PRI_met_sumet","PRI_jet_num","PRI_jet_leading_pt","PRI_jet_leading_eta","PRI_jet_leading_phi","PRI_jet_subleading_pt", "PRI_jet_subleading_eta","PRI_jet_subleading_phi","PRI_jet_all_pt"]].copy().reset_index(drop=True)

Y_V = Validacion["Label"]
W_V = equilibrar(Validacion)

In [ ]:
# Comprobación de estratificación hecha de forma correcta

# 1. Tamaños: el primer número igual en cada grupo
print(Entrenamiento_PRI.shape, Entrenamiento_DER.shape, Y_E.shape, W_E.shape)
print(Validacion_PRI.shape,    Validacion_DER.shape,    Y_V.shape, W_V.shape)

# 2. Labels: solo 0 y 1, con ~34 % de señal
print(np.unique(Y_E), Y_E.mean())     # [0 1]  ~0.34
print(np.unique(Y_V), Y_V.mean())     # [0 1]  ~0.34

# 3. Pesos equilibrados y con media 1
print(W_E[Y_E == 1].sum(), W_E[Y_E == 0].sum())   # iguales
print(W_V[Y_V == 1].sum(), W_V[Y_V == 0].sum())   # iguales
print(W_E.mean(), W_V.mean())                     # 1.0  1.0

# 4. Sin valores que rompan el entrenamiento
for nombre, X in [("Entrenamiento_PRI", Entrenamiento_PRI), ("Entrenamiento_DER", Entrenamiento_DER),
                  ("Validacion_PRI", Validacion_PRI), ("Validacion_DER", Validacion_DER)]:
    print(nombre,
          "NaN:", np.isnan(X).sum(),            # 0
          "inf:", np.isinf(X).sum(),            # 0
          "-999:", (X == -999).sum(),           # 0
          "máx |x|:", np.abs(X).max().round(1)) # no enorme (< 10-20)

# 5. Estandarización (importante para ReLU)
print(Entrenamiento_DER.mean(axis=0).round(2))   # todos cerca de 0
print(Entrenamiento_DER.std(axis=0).round(2))    # todos cerca de 1

(200000, 17) (200000, 31) (200000,) (200000,)
(50000, 17) (50000, 31) (50000,) (50000,)
[0 1] 0.34267
[0 1] 0.34266
99999.99999999997 100000.0
25000.0 24999.999999999996
1.0 1.0
Entrenamiento_PRI NaN: PRI_tau_pt                0
PRI_tau_eta               0
PRI_tau_phi               0
PRI_lep_pt                0
PRI_lep_eta               0
PRI_lep_phi               0
PRI_met                   0
PRI_met_phi               0
PRI_met_sumet             0
PRI_jet_num               0
PRI_jet_leading_pt        0
PRI_jet_leading_eta       0
PRI_jet_leading_phi       0
PRI_jet_subleading_pt     0
PRI_jet_subleading_eta    0
PRI_jet_subleading_phi    0
PRI_jet_all_pt            0
dtype: int64 inf: PRI_tau_pt                0
PRI_tau_eta               0
PRI_tau_phi               0
PRI_lep_pt                0
PRI_lep_eta               0
PRI_lep_phi               0
PRI_met                   0
PRI_met_phi               0
PRI_met_sumet             0
PRI_jet_num               0
PRI_jet_leading_pt       

## Entrenamiento del MLP con ReLU

In [20]:
tf.keras.utils.set_random_seed(42)